In [7]:
import sys
sys.path.append("/var/www/python/Prod/nighthawk/")
import pandas as pd
import numpy as np
import pytz
from datetime import datetime, timedelta

from nighthawk.data.pipeline.constraint_family_pipeline import ConstraintFamilyPipeline, calculate_thresholds_for_ve
from nighthawk.data.product.ve import DailyBidsManager
from nighthawk.util import bigquery_functions


In [8]:
bid_dt = (datetime.now(pytz.timezone('America/Chicago')) + timedelta(days=1)).strftime('%Y-%m-%d')
# bid_dt = '2026-06-04'

bid_dt_minus_3Y = (pd.to_datetime(bid_dt) - pd.Timedelta('1105 day')).strftime('%Y-%m-%d')

print('bid_dt          :', bid_dt)
print('lookback start  :', bid_dt_minus_3Y)


bid_dt          : 2026-06-05
lookback start  : 2023-05-27


## Step 1 — load preautomated cuts portfolio (MISO)

In [9]:
portfolio = DailyBidsManager(opexchange='MISO', bid_date=bid_dt).get_bids_from_table(label='preautomated_cuts')
print('shape          :', portfolio.shape)
print('strategies     :', portfolio['strategy'].unique())
print('date range     :', portfolio['dt'].min(), '->', portfolio['dt'].max())
portfolio.head()


shape          : (8084, 16)
strategies     : ['Fourier' 'hubbleV1' 'DarwinV2' 'HighCap' 'KeplerV2']
date range     : 2026-06-05 -> 2026-06-05


,bid_num,dt,hr,segment,node_num,bid_mw,bid_price,clear_mw,clear_price,cleared_flag,strategy,strategy_id,automated_strategy_id,incdec,submitted_flag,pcid
0,148533691,2026-06-05,1,1,83,0.0,33.08,None,None,None,Fourier,0,0,Decrement,,None
1,148527875,2026-06-05,1,1,129,1.7,16.06,None,None,None,hubbleV1,0,0,Decrement,,None
2,148527876,2026-06-05,1,1,129,1.7,25.80,None,None,None,hubbleV1,0,0,Increment,,None
3,148536391,2026-06-05,1,4,183,10.0,30.25,None,None,None,DarwinV2,0,0,Decrement,,None
4,148540825,2026-06-05,1,4,183,1.5,30.25,None,None,None,HighCap,0,0,Decrement,,None


## Step 2 — initialise ConstraintFamilyPipeline

In [ ]:
pip = ConstraintFamilyPipeline(
    opexchange='MISO',
    start_dt=bid_dt_minus_3Y,
    end_dt=bid_dt,
    mvalue_threshold=100,
    table_suffix='_temp',
    temp=True,
    repopulate_data=True
)
print('pipeline initialised')
print('constraint_family_num_bq :', pip.constraint_family_num_bq)
print('all_dt_bq                :', pip.all_dt_bq)


In [10]:
pip = ConstraintFamilyPipeline(
    opexchange='MISO',
    start_dt=bid_dt_minus_3Y,
    end_dt=bid_dt,
    mvalue_threshold=100,
    table_suffix='_temp',
    temp=True,
    repopulate_data=True
)
print('pipeline initialised')
print('constraint_family_num_bq :', pip.constraint_family_num_bq)
print('all_dt_bq                :', pip.all_dt_bq)


ConstraintFamilyPipeline.get_dart_mvalue_for_constraint_family ran in 28.61s
pipeline initialised
constraint_family_num_bq : constraint_family_exposure.MISO_constraint_family_num_ve_prod
all_dt_bq                : constraint_family_exposure.MISO_all_dt_ve_prod


## Step 3 — get_constraint_family_basic_stats  (DA/RT mvalue, top constraint families)

In [ ]:
pip.get_constraint_family_basic_stats()
print('constraint_family_num_bq :', pip.constraint_family_num_bq)
print('dartm_bq                 :', pip.dartm_bq)

q = f'SELECT * FROM `movetocloud-999.{pip.constraint_family_num_bq}` LIMIT 10'
cf_df = bigquery_functions.download_df_from_bq(q)
print('constraint families found:', len(cf_df))
display(cf_df.head())


## Step 4 — get_dfax  (sensitivity of nodes to each constraint family)

In [ ]:
dfax_bq = pip.get_dfax()
print('dfax_bq :', dfax_bq)

q = f'SELECT * FROM `movetocloud-999.{dfax_bq}` LIMIT 10'
dfax_df = bigquery_functions.download_df_from_bq(q)
print('rows:', len(dfax_df))
display(dfax_df.head())


## Step 5 — get_dayzer_flow  (dayzer predicted DA/RT flow for each constraint family)

In [ ]:
dayzer_bq = pip.get_dayzer_flow()
print('dayzer_table_bq :', dayzer_bq)

q = f'SELECT * FROM `movetocloud-999.{dayzer_bq}` ORDER BY dt DESC LIMIT 20'
dayzer_df = bigquery_functions.download_df_from_bq(q)
print('rows:', len(dayzer_df))
display(dayzer_df.head(10))


## Step 6 — get_constraint_feature  (KV, FlowRatio, monitored/contingency info)

In [ ]:
feat_bq = pip.get_constraint_feature()
print('constraint_features_bq :', feat_bq)

q = f'SELECT * FROM `movetocloud-999.{feat_bq}` LIMIT 10'
feat_df = bigquery_functions.download_df_from_bq(q)
print('rows:', len(feat_df))
display(feat_df.head())


## Step 7 — get_price_derived_variables  (rt_max_in7d, da_max_in7d, da_avg_in30d, adj norms …)

In [ ]:
price_bq = pip.get_price_derived_variables()
print('price_derived_variables_bq :', price_bq)

q = f'SELECT * FROM `movetocloud-999.{price_bq}` ORDER BY dt DESC LIMIT 20'
price_df = bigquery_functions.download_df_from_bq(q)
print('rows:', len(price_df))
display(price_df.head(10))


## Step 8 — get_data  (merge all steps into constraint_family_stats_all)

In [ ]:
stats_bq = pip.get_data(methods_to_include=['dayzer_flow', 'constraint_feature', 'price_derived'])
print('constraint_family_stats_all_bq :', stats_bq)

q = f'SELECT * FROM `movetocloud-999.{stats_bq}` ORDER BY dt DESC LIMIT 20'
stats_df = bigquery_functions.download_df_from_bq(q)
print('rows:', len(stats_df))
print('columns:', stats_df.columns.tolist())
display(stats_df.head(10))


In [ ]:
data_table   = 'constraint_family_exposure.MISO_constraint_family_stats_all_ve_prod'
target_table = 'temp.MISO_constraint_family_exposure_with_quantiles'

calculate_thresholds_for_ve(
    opexchange='MISO',
    data_table=data_table,
    start_dt=bid_dt,
    end_dt=bid_dt,
    target_table=target_table
)
print('quantile table written to:', target_table)

In [ ]:
data_table   = 'constraint_family_exposure.MISO_constraint_family_stats_all_temp'
target_table = 'temp.MISO_constraint_family_exposure_with_quantiles'

calculate_thresholds_for_ve(
    opexchange='MISO',
    data_table=data_table,
    start_dt=bid_dt,
    end_dt=bid_dt,
    target_table=target_table
)
print('quantile table written to:', target_table)


## Step 10 — verify final quantile table

In [ ]:
q = f"""
SELECT *
FROM `movetocloud-999.{target_table}`
WHERE dt = '{bid_dt}'
ORDER BY constraint_family_num
LIMIT 50
"""
result = bigquery_functions.download_df_from_bq(q)
print('rows for', bid_dt, ':', len(result))
print('columns:', result.columns.tolist())
display(result)


---
## run_constraint_family_exposure_risk_cut — MISO (exact flow from risk_manager_ve.py)

In [ ]:
from nighthawk.data.pipeline.var_handler import ice_elec_price_vh
import re

table_suffix = '_ve_prod'   # read from existing prod tables (skip Steps 2-8)
repopulate_data = False

portfolio_bq = bigquery_functions.upload_to_bq_from_dataframe(
    portfolio,
    dataset_name='temp',
    table_name='MISO_portfolio_to_cut',
    temp=True
)
print('portfolio_bq :', portfolio_bq)

start_dt = portfolio['dt'].min()
end_dt   = portfolio['dt'].max()
print('date range   :', start_dt, '->', end_dt)
print('portfolio rows:', len(portfolio))
display(portfolio.head())

In [ ]:
# Skip recomputation — read the exposure table already generated by the prod daily job
constraint_family_exposure_bq = 'constraint_family_exposure.MISO_constraint_family_exposure_ve_prod'
print('constraint_family_exposure_bq :', constraint_family_exposure_bq)

# sanity peek
q = f'SELECT * FROM `movetocloud-999.{constraint_family_exposure_bq}` ORDER BY dt DESC LIMIT 5'
display(bigquery_functions.download_df_from_bq(q))

### RC-3  join exposure with MISO quantiles table — filter mvalue > 100 and short_bid_mw > 0

In [ ]:
bq_query = f""" select a.dt, a.constraint_family_num, oops_constraint_num, monitored_clean, contingency_clean, KV, FlowRatio,
            rt_max_in7d, da_max_in7d, da_avg_in30d, short_bid_mw,
                adj_da_rt_norm_max_q95_0_98_1,
    adj_da_rt_norm_max_q95_0_95_0_98,
    adj_da_rt_norm_max_q95_0_75_0_95,
    adj_da_rt_norm_max_q95_0_0_75,
            from {constraint_family_exposure_bq} as a
            inner join `movetocloud-999.temp.MISO_constraint_family_exposure_with_quantiles` as b
            on a.constraint_family_num = b.constraint_family_num and a.dt = b.dt
             where (da_max_in30d > 100 or rt_max_in30d > 100)  and short_bid_mw > 0"""

all_dt = bigquery_functions.download_df_from_bq(bq_query)
print('all_dt shape:', all_dt.shape)
print('constraint families:', all_dt['constraint_family_num'].nunique())
display(all_dt.head())


In [ ]:
if all_dt.empty:
    print('WARNING: all_dt is empty — no constraints above threshold, no scaling needed')


### RC-4  attach INDIANAHUB ice price

In [ ]:
all_dt['constraint_family_num'] = all_dt['constraint_family_num'].astype(int)

ice_price_var = 'INDIANAHUB_ice_elec_price_forecast_f'
ice_df, _ = ice_elec_price_vh.get_data_and_mapping_for_ice_elec(
    [1656, ], 'MISO', ['INDIANAHUB'], start_dt, end_dt, var_spec=['f'], impute=True
)
ice_df = ice_df.rename(columns={ice_price_var: 'ice_price'})
ice_df = ice_df.groupby('dt').agg({'ice_price': 'mean'}).reset_index()

all_dt = all_dt.merge(ice_df[['dt', 'ice_price']], how='left', on='dt')

print('ice_price range:', all_dt['ice_price'].min(), '->', all_dt['ice_price'].max())
print('nulls in ice_price:', all_dt['ice_price'].isna().sum())
display(ice_df.head())


### RC-5  compute FlowRatio clip and KV_group bins

In [ ]:
all_dt['FlowRatio'] = pd.to_numeric(all_dt['FlowRatio'], errors='coerce').fillna(0)
all_dt['FlowRatio'] = all_dt['FlowRatio'].clip(upper=1)

all_dt['KV'] = pd.to_numeric(all_dt['KV'], errors='coerce')
all_dt['KV_group'] = pd.cut(
    all_dt['KV'],
    bins=[-float('inf'), 1, 115, 138, 345, float('inf')],
    labels=['01_na', '02_<=115', '03_138', '04_345', '05_>=500']
).astype(str)

print('FlowRatio distribution:')
print(all_dt['FlowRatio'].describe())
print('\nKV_group counts:')
print(all_dt['KV_group'].value_counts())


### RC-6  risk_limit rules → normalise by ice_price → multiply by kv_factor

In [ ]:
scale = 3
base_limit = 1_000_000 * scale

risk_limit_rules = [
    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '02_<=115'), base_limit),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '02_<=115'), base_limit),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '02_<=115'), base_limit),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '02_<=115'), base_limit),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '03_138'), base_limit),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '03_138'), base_limit),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '03_138'), base_limit),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '03_138'), base_limit),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '04_345'), base_limit),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '04_345'), base_limit),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '04_345'), base_limit),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '04_345'), base_limit),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '05_>=500'), base_limit),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '05_>=500'), base_limit),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '05_>=500'), base_limit),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '05_>=500'), base_limit),
]

all_dt['risk_limit'] = np.select(
    [c for c, _ in risk_limit_rules],
    [v for _, v in risk_limit_rules],
    default=base_limit
)
all_dt['risk_limit'] = all_dt['risk_limit'] / all_dt['ice_price']

kv = pd.to_numeric(all_dt['KV'], errors='coerce')
kv_clean = kv.mask(kv <= 0)
all_dt['kv_factor'] = np.select(
    [kv_clean.le(69), kv_clean.le(115), kv_clean.le(161), kv_clean.le(220),
     kv_clean.le(345), kv_clean.le(500), kv_clean.gt(500)],
    [0.7, 1.1, 1.6, 2.1, 2.6, 3.1, 3.6],
    default=2.0
)
all_dt['risk_limit'] = all_dt['risk_limit'] * all_dt['kv_factor']

# sanitise column names for BQ upload
all_dt.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', col) for col in all_dt.columns]

print('risk_limit (MW) stats:')
print(all_dt['risk_limit'].describe())
display(all_dt[['constraint_family_num', 'FlowRatio', 'KV_group', 'kv_factor', 'ice_price', 'risk_limit']].head(10))


### RC-7  risk_per_mw rules (historical q95 adj norm by FlowRatio bucket)

In [ ]:
risk_per_mw_rules = [
    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '02_<=115'),
     all_dt['adj_da_rt_norm_max_q95_0_98_1'].apply(lambda x: 46 if pd.isna(x) or x < 46 * 0.2 else x)),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '02_<=115'),
     all_dt['adj_da_rt_norm_max_q95_0_95_0_98'].fillna(26)),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '02_<=115'),
     all_dt['adj_da_rt_norm_max_q95_0_75_0_95'].fillna(20)),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '02_<=115'),
     all_dt['adj_da_rt_norm_max_q95_0_0_75'].fillna(1)),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '03_138'),
     all_dt['adj_da_rt_norm_max_q95_0_98_1'].apply(lambda x: 42 if pd.isna(x) or x < 42 * 0.2 else x)),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '03_138'),
     all_dt['adj_da_rt_norm_max_q95_0_95_0_98'].fillna(28)),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '03_138'),
     all_dt['adj_da_rt_norm_max_q95_0_75_0_95'].fillna(20)),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '03_138'),
     all_dt['adj_da_rt_norm_max_q95_0_0_75'].fillna(3)),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '04_345'),
     all_dt['adj_da_rt_norm_max_q95_0_98_1'].apply(lambda x: 35 if pd.isna(x) or x < 35 * 0.2 else x)),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '04_345'),
     all_dt['adj_da_rt_norm_max_q95_0_95_0_98'].fillna(21)),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '04_345'),
     all_dt['adj_da_rt_norm_max_q95_0_75_0_95'].fillna(17)),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '04_345'),
     all_dt['adj_da_rt_norm_max_q95_0_0_75'].fillna(2)),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '05_>=500'),
     all_dt['adj_da_rt_norm_max_q95_0_98_1'].apply(lambda x: 43 if pd.isna(x) or x < 43 * 0.2 else x)),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '05_>=500'),
     all_dt['adj_da_rt_norm_max_q95_0_95_0_98'].fillna(28)),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '05_>=500'),
     all_dt['adj_da_rt_norm_max_q95_0_75_0_95'].fillna(24)),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '05_>=500'),
     all_dt['adj_da_rt_norm_max_q95_0_0_75'].fillna(1)),
]

all_dt['risk_per_mw'] = np.select(
    [c for c, _ in risk_per_mw_rules],
    [v for _, v in risk_per_mw_rules],
    default=0.1
)

all_dt['risk_per_mw_recent'] = np.where(
    (all_dt['FlowRatio'] > 0.98) | (
        all_dt['FlowRatio'].isna() &
        (all_dt['rt_max_in7d'] > 10 * all_dt['da_avg_in30d']) &
        (all_dt['rt_max_in7d'] > 400)
    ),
    np.maximum(all_dt['rt_max_in7d'] / all_dt['ice_price'] - all_dt['da_max_in7d'] / all_dt['ice_price'], 0.1),
    0.1
)

print('risk_per_mw stats:')
print(all_dt['risk_per_mw'].describe())
print('\nrisk_per_mw_recent stats:')
print(all_dt['risk_per_mw_recent'].describe())
display(all_dt[['constraint_family_num', 'FlowRatio', 'KV_group',
                'risk_per_mw', 'risk_per_mw_recent']].head(10))


### RC-8  long_term_mw_limit, recent_mw_limit, KV_mw_limit

In [ ]:
all_dt['long_term_mw_limit'] = all_dt['risk_limit'] / all_dt['risk_per_mw']
all_dt['recent_mw_limit']    = all_dt['risk_limit'] / all_dt['risk_per_mw_recent']

base_limit = 1000 * scale
all_dt['KV_mw_limit'] = base_limit * all_dt['kv_factor']

print('long_term_mw_limit stats:')
print(all_dt['long_term_mw_limit'].describe())
print('\nrecent_mw_limit stats:')
print(all_dt['recent_mw_limit'].describe())
print('\nKV_mw_limit stats:')
print(all_dt['KV_mw_limit'].describe())
display(all_dt[['constraint_family_num', 'short_bid_mw',
                'long_term_mw_limit', 'recent_mw_limit', 'KV_mw_limit']].head(10))


### RC-9  scale factors — long_term / recent / KV → final_factor = min of three

In [ ]:
all_dt['long_term_risk_scale_factor'] = 1.0
all_dt['recent_risk_scale_factor']    = 1.0
all_dt['kv_mw_scale_factor']          = 1.0

nonzero_mask = all_dt['short_bid_mw'] != 0

all_dt.loc[nonzero_mask, 'long_term_risk_scale_factor'] = (
    np.minimum(all_dt.loc[nonzero_mask, 'short_bid_mw'], all_dt.loc[nonzero_mask, 'long_term_mw_limit'])
    / all_dt.loc[nonzero_mask, 'short_bid_mw']
)
all_dt.loc[nonzero_mask, 'recent_risk_scale_factor'] = (
    np.minimum(all_dt.loc[nonzero_mask, 'short_bid_mw'], all_dt.loc[nonzero_mask, 'recent_mw_limit'])
    / all_dt.loc[nonzero_mask, 'short_bid_mw']
)
all_dt.loc[nonzero_mask, 'kv_mw_scale_factor'] = (
    np.minimum(all_dt.loc[nonzero_mask, 'short_bid_mw'], all_dt.loc[nonzero_mask, 'KV_mw_limit'])
    / all_dt.loc[nonzero_mask, 'short_bid_mw']
)

all_dt['final_factor'] = all_dt[
    ['long_term_risk_scale_factor', 'recent_risk_scale_factor', 'kv_mw_scale_factor']
].min(axis=1)

print('constraints being cut (final_factor < 1):', (all_dt['final_factor'] < 1).sum())
print('\nfinal_factor distribution:')
print(all_dt['final_factor'].describe())
display(all_dt[['constraint_family_num', 'short_bid_mw',
                'long_term_risk_scale_factor', 'recent_risk_scale_factor',
                'kv_mw_scale_factor', 'final_factor']].head(15))


In [ ]:
scale_table = bigquery_functions.upload_to_bq_from_dataframe(
    all_dt[['dt', 'constraint_family_num', 'monitored_clean', 'KV_group', 'final_factor']],
    'temp',
    'MISO_scale_table_temp',
    temp=True
)
print('scale_table :', scale_table)

q = f'SELECT * FROM `movetocloud-999.{scale_table}` LIMIT 10'
display(bigquery_functions.download_df_from_bq(q))


In [ ]:
scale_table = bigquery_functions.upload_to_bq_from_dataframe(
    all_dt[['dt', 'constraint_family_num', 'monitored_clean', 'KV_group', 'final_factor']],
    'temp',
    'MISO_scale_table_temp',
    temp=True
)
print('scale_table :', scale_table)

q = f'SELECT * FROM `movetocloud-999.{scale_table}` LIMIT 10'
display(bigquery_functions.download_df_from_bq(q))


### RC-11  apply scale factors via BQ — join MISO_dfax_ve_prod → scaled_bid_mw

In [ ]:
bq_query = f"""
WITH hr_table AS (
  SELECT x AS hr FROM UNNEST(GENERATE_ARRAY(1, 24)) AS x
),

factor_table AS (
  SELECT f.dt, f.node_num, h.hr,
         MIN(f.inc_factor) AS inc_factor,
         MIN(f.dec_factor) AS dec_factor
  FROM (
    SELECT dt, a.constraint_family_num, node_num, dfax, final_factor AS factor,
        CASE
          -- reverse dfax since we are doing short cut
          WHEN dfax > 0 AND KV_group IN (\"02_<=115\", \"03_138\", \"04_345\", \"05_>=500\") THEN final_factor
          ELSE 1
        END AS inc_factor,
        CASE
          WHEN dfax < 0 AND KV_group IN (\"02_<=115\", \"03_138\", \"04_345\", \"05_>=500\") THEN final_factor
          ELSE 1
        END AS dec_factor
    FROM `movetocloud-999.{scale_table}` AS a
    LEFT JOIN `movetocloud-999.constraint_family_exposure.MISO_dfax_ve_prod` AS b
      ON a.constraint_family_num = b.constraintFamilyNum
    WHERE ABS(dfax) > 0.05
  ) AS f
  CROSS JOIN hr_table AS h
  GROUP BY f.dt, f.node_num, h.hr
)

SELECT
  M.* EXCEPT(dt),
  CAST(M.dt AS STRING) AS dt,
  CASE
    WHEN M.incdec = 'Increment' THEN COALESCE(M.bid_mw * N.inc_factor, M.bid_mw)
    ELSE COALESCE(M.bid_mw * N.dec_factor, M.bid_mw)
  END AS scaled_bid_mw

FROM `{portfolio_bq}` AS M

-- inner join ensures only dates present in scale_table are included
INNER JOIN (
  SELECT DISTINCT dt FROM `movetocloud-999.{scale_table}`
) AS Q ON CAST(M.dt AS STRING) = Q.dt

LEFT JOIN factor_table AS N
  ON CAST(M.dt AS STRING) = N.dt AND M.hr = N.hr AND M.node_num = N.node_num
"""

portfolio_scaled = bigquery_functions.download_df_from_bq(bq_query)
portfolio_scaled['bid_mw_original'] = portfolio_scaled['bid_mw']
portfolio_scaled['bid_mw'] = portfolio_scaled['scaled_bid_mw']

print('portfolio_scaled shape:', portfolio_scaled.shape)
display(portfolio_scaled.head())


### RC-12  final summary — bid_mw reduction by incdec and top affected nodes

In [ ]:
diff = portfolio_scaled.groupby('incdec').agg(
    bid_mw_original=('bid_mw_original', 'sum'),
    bid_mw_scaled=('bid_mw', 'sum')
).reset_index()
diff['reduction_pct'] = (1 - diff['bid_mw_scaled'] / diff['bid_mw_original']) * 100
print('Overall reduction by incdec:')
display(diff)

node_diff = portfolio_scaled.groupby(['node_num', 'incdec']).agg(
    bid_mw_original=('bid_mw_original', 'sum'),
    bid_mw_scaled=('bid_mw', 'sum')
).reset_index()
node_diff['reduction'] = node_diff['bid_mw_original'] - node_diff['bid_mw_scaled']
print('\nTop 20 most affected nodes:')
display(node_diff.sort_values('reduction', ascending=False).head(20))
